<a href="https://colab.research.google.com/github/BartoszKalinowski1/bachelor-thesis/blob/feature_models_comparison/kumo_tabred_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kumo Tabular na danych TabReD (Google Colab)

Notebook pobiera zbiór z benchmarku TabReD, dopasowuje go do ograniczeń Kumo Tabular i porównuje go z LightGBM na podziale czasowym.

**Kolejność:** uruchamiaj komórki od góry. Zalecany pierwszy przebieg: `sberbank-housing` (mały, regresja), potem `homecredit-default` (klasyfikacja, finanse).

| Zbiór | Cech | Zadanie | Wierszy użytych |
|---|---|---|---|
| homesite-insurance | 299 | klasyfikacja | 260 753 |
| ecom-offers | 119 | klasyfikacja | 160 057 |
| homecredit-default | 696 | klasyfikacja | 381 664 |
| sberbank-housing | 392 | regresja | 28 321 |
| cooking-time | 192 | regresja | 319 986 |
| delivery-eta | 223 | regresja | 416 451 |
| maps-routing | 986 | regresja | 340 981 |
| weather | 103 | regresja | 423 795 |

Nazwy do pobierania: dla `cooking-time` i `weather` pochodzą wprost ze strony TabReD, pozostałe wywnioskowałem ze wzoru nazw. Jeśli komenda zgłosi błąd, sprawdź `uvx tabred download --help`.

**Ważne ograniczenie:** Kumo Tabular był pretrenowany na tabelach do ok. 60 tys. wierszy i 100 cech (2-10 klas). TabReD ma 103-986 cech, więc notebook wybiera 100 cech (tylko na podstawie zbioru treningowego) i ogranicza kontekst. Wyniki nie są więc porównywalne z liczbami z artykułu TabReD i trzeba to zaznaczyć w pracy.

## 1. Instalacja

In [ ]:
!pip -q install structured-data-models scikit-learn lightgbm uv

In [ ]:
# Instalacja brakujących bibliotek XGBoost i CatBoost do porównania GBDT
!pip -q install xgboost catboost

In [ ]:
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
# Potrzebne: Python >= 3.11 i PyTorch >= 2.7. Jesli wersja jest nizsza, zrestartuj sesje lub zaktualizuj torch.

3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
torch 2.14.1+cpu | cuda: False


## 2. Uwierzytelnienie w Kaggle
Wygeneruj token: Kaggle → Settings → API → Create New Token (pobierze się `kaggle.json`). Dla zbiorow oznaczonych na stronie TabReD jako *Competition* moze byc konieczne wczesniejsze zaakceptowanie regulaminu konkursu na Kaggle (nie sprawdzalem tego).

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    token = getpass.getpass("Wklej Kaggle API token (KGAT_...): ")

os.environ["KAGGLE_API_TOKEN"] = token
kdir = os.path.expanduser("~/.kaggle")
os.makedirs(kdir, exist_ok=True)
path = os.path.join(kdir, "access_token")
with open(path, "w") as f:
    f.write(token)
os.chmod(path, 0o600)
del token
print("Token ustawiony.")

Token ustawiony.


In [ ]:
# Poprawka konfiguracji Kaggle API
import os

try:
    from google.colab import userdata
    token_val = userdata.get("KAGGLE_API_TOKEN")
    # Alternatywnie sprawdźmy powszechną nazwę klucza
    if not token_val:
        token_val = userdata.get("KAGGLE_KEY")
except Exception:
    token_val = None

if not token_val:
    token_val = os.environ.get("KAGGLE_API_TOKEN")

if token_val:
    # Zapisz do standardowego pliku wymaganej przez oficjalne API kaggle.json
    os.environ["KAGGLE_USERNAME"] = "" # Jeśli masz pełny plik json, upewnij się że podajesz poprawny format.
    # Najbezpieczniej ustawić bezpośrednią zmienną API Token dla biblioteki pythonowej
    os.environ["KAGGLE_API_TOKEN"] = token_val

    kdir = os.path.expanduser("~/.kaggle")
    os.makedirs(kdir, exist_ok=True)

    # Niektóre wersje wymagają struktury JSON
    import json
    # Jeśli token_val to sam klucz, a nazwa użytkownika jest opcjonalna:
    if "{" not in token_val:
        # Próba zapisu standardowego klucza tekstowego
        with open(os.path.join(kdir, "kaggle.json"), "w") as f:
            # Jeśli token_val zawiera format '{"username":"...","key":"..."}', zapiszemy go bezpośrednio
            f.write(token_val)
    else:
        with open(os.path.join(kdir, "kaggle.json"), "w") as f:
            f.write(token_val)

    os.chmod(os.path.join(kdir, "kaggle.json"), 0o600)
    print("Poprawiono i zapisano plik kaggle.json w ~/.kaggle/")
else:
    print("OSTRZEŻENIE: Nie odnaleziono tokenu Kaggle w Secrets ani w zmiennych środowiskowych. Upewnij się, że dodałeś KAGGLE_API_TOKEN do sekretów Colaba!")

Poprawiono i zapisano plik kaggle.json w ~/.kaggle/


## 3. Konfiguracja i pobranie danych

In [ ]:
# Krok 3: Konfiguracja i ponowne pobranie danych po poprawieniu tokenu Kaggle
DATASET = "sberbank-housing"      # patrz tabela powyzej
TASK = "regression"               # "classification" albo "regression"
SPLIT = "default"                 # "default", "random-0", ...
OUT = "/content/tabred"

# Pobieranie danych TabReD
print(f"Pobieranie zbioru {DATASET}...")
!uvx --with six tabred download {DATASET} --output-path {OUT}
print("Pobieranie zakończone. Możesz teraz przejść do kroku 4 (wczytanie danych).")

Pobieranie zbioru sberbank-housing...
⠴ Checking Kaggle competition access
✓ competition access verified
✓ unpacked requested data into `/content/tabred`
Pobieranie zakończone. Możesz teraz przejść do kroku 4 (wczytanie danych).


## 4. Wczytanie danych

In [ ]:
import json, glob, numpy as np, pandas as pd

hits = glob.glob(f"{OUT}/**/{DATASET}/info.json", recursive=True) or glob.glob(f"{OUT}/**/info.json", recursive=True)
assert hits, "Nie znaleziono info.json - sprawdz czy pobieranie sie powiodlo"
root = os.path.dirname(hits[0])
print("Katalog:", root)
print(open(hits[0]).read()[:600])
print(sorted(os.listdir(root)))

def load(name):
    p = f"{root}/{name}.npy"
    return np.load(p, allow_pickle=True) if os.path.exists(p) else None

xn, xc, xb, xm, y = load("x_num"), load("x_cat"), load("x_bin"), load("x_meta"), load("y")

cols = {}
if xn is not None:
    for i in range(xn.shape[1]): cols[f"num_{i}"] = xn[:, i].astype("float32")
if xb is not None:
    for i in range(xb.shape[1]): cols[f"bin_{i}"] = xb[:, i].astype("float32")
if xc is not None:
    for i in range(xc.shape[1]): cols[f"cat_{i}"] = pd.factorize(xc[:, i])[0].astype("int32")
X = pd.DataFrame(cols)
y = y.astype(int) if TASK == "classification" else y.astype("float32")
print("X:", X.shape, "| y:", y.shape)

tr = np.load(f"{root}/splits/{SPLIT}/train.npy")
va = np.load(f"{root}/splits/{SPLIT}/val.npy")
te = np.load(f"{root}/splits/{SPLIT}/test.npy")
print("train/val/test:", len(tr), len(va), len(te))
if xm is not None:
    print("x_meta ksztalt:", xm.shape, "(sprawdz, czy to znaczniki czasu)")

Katalog: /content/tabred/sberbank-housing
{
    "task": {
        "type": "regression",
        "score": "rmse"
    }
}

['info.json', 'splits', 'x_bin.npy', 'x_cat.npy', 'x_meta.npy', 'x_num.npy', 'y.npy']
X: (28321, 392) | y: (28321,)
train/val/test: 18847 4827 4647
x_meta ksztalt: (28321, 1) (sprawdz, czy to znaczniki czasu)


## 5. Wybor 100 cech (tylko ze zbioru treningowego)
Selekcja przez waznosc cech z LightGBM na probce treningowej, bez uzycia danych walidacyjnych i testowych (zeby nie bylo wycieku).

In [ ]:
import lightgbm as lgb

MAX_FEATURES = 100
rng = np.random.default_rng(0)

if X.shape[1] > MAX_FEATURES:
    sel = rng.choice(tr, size=min(20000, len(tr)), replace=False)
    Est = lgb.LGBMClassifier if TASK == "classification" else lgb.LGBMRegressor
    m = Est(n_estimators=200, learning_rate=0.1, importance_type="gain", verbose=-1, random_state=0)
    m.fit(X.iloc[sel], y[sel])
    imp = pd.Series(m.feature_importances_, index=X.columns).sort_values(ascending=False)
    features = list(imp.index[:MAX_FEATURES])
else:
    features = list(X.columns)
print("Liczba cech:", len(features))

Liczba cech: 100


## 6. Kontekst i zapytania dla Kumo
- `CONTEXT_N`: ile wierszy treningowych trafia do kontekstu (pretrening do ok. 60 tys.). Zacznij od 10 000, zwiekszaj na GPU.
- `CONTEXT_MODE = "recent"`: ostatnie wiersze zbioru treningowego (zakladam, ze kolejnosc wierszy jest chronologiczna, zweryfikuj w `x_meta`/`info.json`). `"random"`: losowa probka. Porownanie obu to gotowy eksperyment o doborze kontekstu przy dryfie.
- `N_TEST`: probka zbioru testowego (rownomiernie rozlozona w czasie), zeby inferencja byla szybka.

In [ ]:
CONTEXT_N = 10_000
N_TEST = 20_000
CONTEXT_MODE = "recent"     # "recent" albo "random"

tr_sorted, te_sorted = np.sort(tr), np.sort(te)
if CONTEXT_MODE == "recent":
    ctx_idx = tr_sorted[-CONTEXT_N:]
else:
    ctx_idx = np.sort(rng.choice(tr, size=min(CONTEXT_N, len(tr)), replace=False))
if len(te_sorted) > N_TEST:
    te_idx = te_sorted[np.linspace(0, len(te_sorted) - 1, N_TEST).astype(int)]
else:
    te_idx = te_sorted
print("kontekst:", len(ctx_idx), "| test:", len(te_idx))

kontekst: 10000 | test: 4647


## 7. Predykcje Kumo Tabular

In [ ]:
import sdm

device = "cuda" if torch.cuda.is_available() else "cpu"
SIZE = "small"              # na GPU mozna uzyc domyslnego duzego modelu (usun size=...)

def make_df(idx):
    d = X.iloc[idx][features].copy()
    d["target"] = y[idx]
    return d.reset_index(drop=True)

df_ctx, df_te = make_df(ctx_idx), make_df(te_idx)
df_all = pd.concat([df_ctx, df_te], ignore_index=True)

overrides = {c: "categorical" for c in features if c.startswith("cat_")}
if TASK == "classification":
    overrides["target"] = "categorical"
stypes = sdm.infer_stypes(df_all, overrides=overrides)

table = sdm.TableTensor.from_pandas(df=df_all, stypes=stypes, device=device)
context, query = table[:len(df_ctx)], table[len(df_ctx):]

model = sdm.models.KumoTabular(task=TASK, size=SIZE, device=device)

CHUNK = 2000
outs = []
with torch.no_grad():
    for xq in query.drop_columns("target").split(CHUNK):
        kw = dict(
            x_context=context.drop_columns("target"),
            y_context=context["target"],
            x_query=xq,
            generator=torch.Generator(device=device).manual_seed(0),
        )
        if TASK == "classification":
            kw["num_estimators"] = 8
        outs.append(model(**kw))
print("gotowe, partii:", len(outs))

config.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

small/regressor.pt: reconstructing file:   0%|          |  0.00B /  114MB            

small/regressor.pt: downloading bytes:           |  0.00B            

gotowe, partii: 3


## 8. Metryki

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss

y_true = df_te["target"].values
results = {}

def report(name, pred, lo=None, hi=None):
    if TASK == "classification":
        p = np.clip(pred, 1e-6, 1 - 1e-6)
        results[name] = {"AUC": roc_auc_score(y_true, p), "LogLoss": log_loss(y_true, p)}
    else:
        r = {"RMSE": float(np.sqrt(np.mean((pred - y_true) ** 2))), "MAE": float(np.mean(np.abs(pred - y_true)))}
        if lo is not None:
            r["Pokrycie 90%"] = float(np.mean((y_true >= lo) & (y_true <= hi)))
        results[name] = r

if TASK == "classification":
    probs = pd.concat([o.to_pandas() for o in outs], ignore_index=True)
    probs.columns = probs.columns.astype(str)
    report("Kumo Tabular", probs["1"].values)
else:
    def col(q):
        return np.concatenate([o[q].numerical.squeeze(-1).cpu().numpy() for o in outs])
    report("Kumo Tabular", col("q500"), col("q050"), col("q950"))

pd.DataFrame(results).T

,RMSE,MAE,Pokrycie 90%
Kumo Tabular,0.235923,0.143293,0.978696


## 9. Punkt odniesienia: LightGBM
Dwa warianty: (a) na **tych samych** wierszach co kontekst Kumo (rowne dane), (b) na **calym** zbiorze treningowym z wczesnym zatrzymaniem na walidacji (realistyczny). Parametry domyslne, do pracy trzeba je stroic.

```markdown
## 9. Rozszerzone porównanie: LightGBM, XGBoost, CatBoost oraz Zapis Wyników

Poniższy kod uruchamia pełny potok ewaluacyjny dla aktualnie pobranego zbioru danych. Testujemy trzy reżimy:
1. **Kumo Tabular** (na wybranym kontekście `CONTEXT_N`)
2. **GBDT (Ten sam kontekst)**: LightGBM, XGBoost, CatBoost trenowane wyłącznie na tych samych wierszach co kontekst Kumo.
3. **GBDT (Pełny train)**: GBDT trenowane na całym zbiorze treningowym z early stopping na zbiorze walidacyjnym.

Dla zadań regresji implementujemy również **regresję kwantylową** (kwantyle 0.05 i 0.95), aby zmierzyć pokrycie przedziału 90% i bezpośrednio porównać je z Kumo.
```

In [ ]:
import time
import numpy as np
import pandas as pd
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score, log_loss

# Upewnienie się, że dane i zmienne globalne są załadowane z poprzednich kroków
try:
    _ = X, y, tr, va, features, df_te, ctx_idx
except NameError:
    raise NameError("Błąd: Najpierw musisz uruchomić poprzednie komórki (kroki 1 do 8), aby załadować i przygotować dane!")

# Przygotowanie struktur danych dla modeli GBDT
X_train_full = X.iloc[tr][features]
y_train_full = y[tr]
X_val_full = X.iloc[va][features]
y_val_full = y[va]

X_ctx = X.iloc[ctx_idx][features]
y_ctx = y[ctx_idx]

X_test = df_te[features]
y_test = df_te["target"].values

benchmark_records = []

def add_record(model_name, regime, metrics, train_time, pred_time):
    rec = {
        "Dataset": DATASET,
        "Split": SPLIT,
        "Model": model_name,
        "Regime": regime,
        "Train_Time_Sec": round(train_time, 2),
        "Prediction_Time_Sec": round(pred_time, 2)
    }
    rec.update(metrics)
    benchmark_records.append(rec)
    print(f"[{model_name} - {regime}] Zebrane metryki: {metrics}")

# --- 1. Zapisujemy wyniki Kumo Tabular z poprzednich kroków ---
kumo_metrics = results.get("Kumo Tabular", {})
add_record("Kumo Tabular", "context_only", kumo_metrics, 0.0, 0.0)

# --- 2. Ewaluacja GBDT: Reżim "Ten sam kontekst" vs "Pełny train" ---
if TASK == "regression":
    # Funkcja pomocnicza do liczenia pokrycia przedziału ufności 90%
    def calc_coverage(y_true, q05, q95):
        return float(np.mean((y_true >= q05) & (y_true <= q95)))

    # --- LIGHTGBM ---
    # Reżim: Ten sam kontekst (punktowy + kwantyle)
    t0 = time.time()
    lgb_pt = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0).fit(X_ctx, y_ctx)
    lgb_q05 = lgb.LGBMRegressor(objective='quantile', alpha=0.05, n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0).fit(X_ctx, y_ctx)
    lgb_q95 = lgb.LGBMRegressor(objective='quantile', alpha=0.95, n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0).fit(X_ctx, y_ctx)
    t_train = time.time() - t0

    t1 = time.time()
    pred_pt = lgb_pt.predict(X_test)
    pred_q05 = lgb_q05.predict(X_test)
    pred_q95 = lgb_q95.predict(X_test)
    t_pred = time.time() - t1

    metrics = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred_pt))),
        "MAE": float(mean_absolute_error(y_test, pred_pt)),
        "Pokrycie 90%": calc_coverage(y_test, pred_q05, pred_q95)
    }
    add_record("LightGBM", "context_only", metrics, t_train, t_pred)

    # Reżim: Pełny trening z walidacją
    t0 = time.time()
    lgb_full = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, verbose=-1, random_state=0)
    lgb_full.fit(X_train_full, y_train_full, eval_set=[(X_val_full, y_val_full)], callbacks=[lgb.early_stopping(50, verbose=False)])
    t_train = time.time() - t0

    t1 = time.time()
    pred_full = lgb_full.predict(X_test)
    t_pred = time.time() - t1

    metrics_full = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred_full))),
        "MAE": float(mean_absolute_error(y_test, pred_full))
    }
    add_record("LightGBM", "full_train", metrics_full, t_train, t_pred)

    # --- XGBOOST ---
    # Reżim: Ten sam kontekst
    t0 = time.time()
    xgb_pt = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=0).fit(X_ctx, y_ctx)
    t_train = time.time() - t0

    t1 = time.time()
    pred_pt = xgb_pt.predict(X_test)
    t_pred = time.time() - t1
    metrics = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred_pt))),
        "MAE": float(mean_absolute_error(y_test, pred_pt))
    }
    add_record("XGBoost", "context_only", metrics, t_train, t_pred)

    # Reżim: Pełny trening
    t0 = time.time()
    xgb_full = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.05, random_state=0)
    xgb_full.fit(X_train_full, y_train_full, eval_set=[(X_val_full, y_val_full)], verbose=False)
    t_train = time.time() - t0

    t1 = time.time()
    pred_full = xgb_full.predict(X_test)
    t_pred = time.time() - t1
    metrics_full = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred_full))),
        "MAE": float(mean_absolute_error(y_test, pred_full))
    }
    add_record("XGBoost", "full_train", metrics_full, t_train, t_pred)

    # --- CATBOOST ---
    # Reżim: Ten sam kontekst
    t0 = time.time()
    cb_pt = cb.CatBoostRegressor(iterations=300, learning_rate=0.05, verbose=0, random_state=0).fit(X_ctx, y_ctx)
    t_train = time.time() - t0

    t1 = time.time()
    pred_pt = cb_pt.predict(X_test)
    t_pred = time.time() - t1
    metrics = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred_pt))),
        "MAE": float(mean_absolute_error(y_test, pred_pt))
    }
    add_record("CatBoost", "context_only", metrics, t_train, t_pred)

    # Reżim: Pełny trening
    t0 = time.time()
    cb_full = cb.CatBoostRegressor(iterations=2000, learning_rate=0.05, verbose=0, random_state=0)
    cb_full.fit(X_train_full, y_train_full, eval_set=(X_val_full, y_val_full), early_stopping_rounds=50, verbose=0)
    t_train = time.time() - t0

    t1 = time.time()
    pred_full = cb_full.predict(X_test)
    t_pred = time.time() - t1
    metrics_full = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred_full))),
        "MAE": float(mean_absolute_error(y_test, pred_full))
    }
    add_record("CatBoost", "full_train", metrics_full, t_train, t_pred)

else:
    # --- KLASYFIKACJA ---
    # --- LIGHTGBM ---
    # Reżim: Ten sam kontekst
    t0 = time.time()
    lgb_pt = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0).fit(X_ctx, y_ctx)
    t_train = time.time() - t0

    t1 = time.time()
    pred_pt = np.clip(lgb_pt.predict_proba(X_test)[:, 1], 1e-6, 1 - 1e-6)
    t_pred = time.time() - t1
    metrics = {
        "AUC": float(roc_auc_score(y_test, pred_pt)),
        "LogLoss": float(log_loss(y_test, pred_pt))
    }
    add_record("LightGBM", "context_only", metrics, t_train, t_pred)

    # Reżim: Pełny trening
    t0 = time.time()
    lgb_full = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, verbose=-1, random_state=0)
    lgb_full.fit(X_train_full, y_train_full, eval_set=[(X_val_full, y_val_full)], callbacks=[lgb.early_stopping(50, verbose=False)])
    t_train = time.time() - t0

    t1 = time.time()
    pred_full = np.clip(lgb_full.predict_proba(X_test)[:, 1], 1e-6, 1 - 1e-6)
    t_pred = time.time() - t1
    metrics_full = {
        "AUC": float(roc_auc_score(y_test, pred_full)),
        "LogLoss": float(log_loss(y_test, pred_full))
    }
    add_record("LightGBM", "full_train", metrics_full, t_train, t_pred)

    # --- XGBOOST ---
    # Reżim: Ten sam kontekst
    t0 = time.time()
    xgb_pt = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=0).fit(X_ctx, y_ctx)
    t_train = time.time() - t0

    t1 = time.time()
    pred_pt = np.clip(xgb_pt.predict_proba(X_test)[:, 1], 1e-6, 1 - 1e-6)
    t_pred = time.time() - t1
    metrics = {
        "AUC": float(roc_auc_score(y_test, pred_pt)),
        "LogLoss": float(log_loss(y_test, pred_pt))
    }
    add_record("XGBoost", "context_only", metrics, t_train, t_pred)

    # Reżim: Pełny trening
    t0 = time.time()
    xgb_full = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, random_state=0)
    xgb_full.fit(X_train_full, y_train_full, eval_set=[(X_val_full, y_val_full)], verbose=False)
    t_train = time.time() - t0

    t1 = time.time()
    pred_full = np.clip(xgb_full.predict_proba(X_test)[:, 1], 1e-6, 1 - 1e-6)
    t_pred = time.time() - t1
    metrics_full = {
        "AUC": float(roc_auc_score(y_test, pred_full)),
        "LogLoss": float(log_loss(y_test, pred_full))
    }
    add_record("XGBoost", "full_train", metrics_full, t_train, t_pred)

    # --- CATBOOST ---
    # Reżim: Ten sam kontekst
    t0 = time.time()
    cb_pt = cb.CatBoostClassifier(iterations=300, learning_rate=0.05, verbose=0, random_state=0).fit(X_ctx, y_ctx)
    t_train = time.time() - t0

    t1 = time.time()
    pred_pt = np.clip(cb_pt.predict_proba(X_test)[:, 1], 1e-6, 1 - 1e-6)
    t_pred = time.time() - t1
    metrics = {
        "AUC": float(roc_auc_score(y_test, pred_pt)),
        "LogLoss": float(log_loss(y_test, pred_pt))
    }
    add_record("CatBoost", "context_only", metrics, t_train, t_pred)

    # Reżim: Pełny trening
    t0 = time.time()
    cb_full = cb.CatBoostClassifier(iterations=2000, learning_rate=0.05, verbose=0, random_state=0)
    cb_full.fit(X_train_full, y_train_full, eval_set=(X_val_full, y_val_full), early_stopping_rounds=50, verbose=0)
    t_train = time.time() - t0

    t1 = time.time()
    pred_full = np.clip(cb_full.predict_proba(X_test)[:, 1], 1e-6, 1 - 1e-6)
    t_pred = time.time() - t1
    metrics_full = {
        "AUC": float(roc_auc_score(y_test, pred_full)),
        "LogLoss": float(log_loss(y_test, pred_full))
    }
    add_record("CatBoost", "full_train", metrics_full, t_train, t_pred)

[Kumo Tabular - context_only] Zebrane metryki: {'RMSE': 0.2359229475259781, 'MAE': 0.14329326152801514, 'Pokrycie 90%': 0.9786959328599096}
[LightGBM - context_only] Zebrane metryki: {'RMSE': 0.24432150176557002, 'MAE': 0.1606514106146681, 'Pokrycie 90%': 0.7609210243167635}
[LightGBM - full_train] Zebrane metryki: {'RMSE': 0.24755615938675257, 'MAE': 0.16559824824389016}
[XGBoost - context_only] Zebrane metryki: {'RMSE': 0.2910029645380443, 'MAE': 0.20598168671131134}
[XGBoost - full_train] Zebrane metryki: {'RMSE': 0.38580385511938203, 'MAE': 0.30060839653015137}
[CatBoost - context_only] Zebrane metryki: {'RMSE': 0.24544767181111582, 'MAE': 0.16296122012748612}
[CatBoost - full_train] Zebrane metryki: {'RMSE': 0.24813508455638322, 'MAE': 0.16868495063446723}


In [ ]:
import os
import pandas as pd

# Upewnienie się, że tabela wyników istnieje
try:
    df_results = pd.DataFrame(benchmark_records)
except NameError:
    raise NameError("Błąd: Musisz najpierw uruchomić komórkę z ewaluacją modeli (krok 9), aby wygenerować zmienną 'benchmark_records'!")

# Zapis do CSV (lokalnie, do pobrania lub eksportu na Dysk Google)
csv_path = "tabred_benchmark_results.csv"
df_results.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
print(f"Wyniki zapisano do: {csv_path}")

display(df_results)

Wyniki zapisano do: tabred_benchmark_results.csv


,Dataset,Split,Model,Regime,Train_Time_Sec,Prediction_Time_Sec,RMSE,MAE,Pokrycie 90%
0,sberbank-housing,default,Kumo Tabular,context_only,0.00,0.00,0.235923,0.143293,0.978696
1,sberbank-housing,default,LightGBM,context_only,16.74,0.37,0.244322,0.160651,0.760921
2,sberbank-housing,default,LightGBM,full_train,5.42,0.07,0.247556,0.165598,NaN
3,sberbank-housing,default,XGBoost,context_only,7.09,0.05,0.291003,0.205982,NaN
4,sberbank-housing,default,XGBoost,full_train,103.11,0.26,0.385804,0.300608,NaN
5,sberbank-housing,default,CatBoost,context_only,8.73,0.02,0.245448,0.162961,NaN
6,sberbank-housing,default,CatBoost,full_train,13.85,0.01,0.248135,0.168685,NaN


In [ ]:
Est = lgb.LGBMClassifier if TASK == "classification" else lgb.LGBMRegressor
Xte = df_te[features]

def fit_predict(train_idx, use_val):
    mdl = Est(n_estimators=2000 if use_val else 300, learning_rate=0.05, verbose=-1, random_state=0)
    Xtr, ytr = X.iloc[train_idx][features], y[train_idx]
    if use_val:
        mdl.fit(Xtr, ytr, eval_set=[(X.iloc[va][features], y[va])], callbacks=[lgb.early_stopping(50, verbose=False)])
    else:
        mdl.fit(Xtr, ytr)
    return mdl.predict_proba(Xte)[:, 1] if TASK == "classification" else mdl.predict(Xte)

report("LightGBM (ten sam kontekst)", fit_predict(ctx_idx, use_val=False))
report("LightGBM (caly train)", fit_predict(tr, use_val=True))

pd.DataFrame(results).T

,RMSE,MAE,Pokrycie 90%
Kumo Tabular,0.235923,0.143293,0.978696
LightGBM (ten sam kontekst),0.244322,0.160651,NaN
LightGBM (caly train),0.247556,0.165598,NaN
